# Imports and setups

In [19]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import pickle
import matplotlib.cm as cm
import os

from test_utils import load_resultData, load_model_from_state

In [20]:
# Pick file, activation function and seed to load results from
file_dir = "main_UAV"
activation_dir = "tanh_tanh"        # "softplus_tanh", "tanh_softplus"
SEEDS = [0, 1]

results_dir = "results"
data_dir = "resultData.pkl"
file_name = f"{file_dir}_recreation"

models_dict = {}
history_dict = {}
time_dict = {}
indices_dict = {}
fisher_scores_dict = {}

for s, seed in enumerate(SEEDS):
    seed_dir = f"seed_{seed}"
    load_dir = os.path.join(results_dir, file_dir, activation_dir, seed_dir, data_dir)

    SEED, models_state, histories, indices, fisher_scores, timings, config = load_resultData(load_dir)

    learning_percentages = config["learning_percentages"]
    eta = config["eta"]
    eta_min = config["eta_min"]
    Weight_decay = config["Weight_decay"]
    hidden_layers_size = config["hidden_layers_size"]
    activation_functions = config["activation_functions"]
    n_input = config["n_input"]
    n_output = config["n_output"]
    p = config["p"]
    fisher_training_epochs = config["fisher_training_epochs"]
    budget = config["budget"]
    b = config["b"]
    epochs = config["epochs"]
    use_OFE = config["use_OFE"]
    train_epochs = config["train_epochs"]
    num_bins_for_curriculum = config["num_bins_for_curriculum"]
    save_dir = config["save_dir"]

    activation_fn = activation_functions[0]  # Activation function used in 
    train_activation_fn = activation_functions[1]  # Activation function used during training for calculating the loss (can be different from the one used in the model definition, but in this case we will use the same)

    models_dict[SEED] = {}
    for key in models_state.keys():
        models_dict[SEED][key] = {}
        # print(f"Loading models for {key}...")
        for method, state in models_state[key].items():
            # print(f"Loading model for method: {method}...")
            if state is not None:
                models_dict[SEED][key][method] = load_model_from_state(state, config)

    history_dict[SEED] = histories

    indices_dict[SEED] = indices

    time_dict[SEED] = timings

    fisher_scores_dict[SEED] = fisher_scores

NN_config = {
    "n_input": n_input,
    "n_output": n_output,
    "hidden_layers_size": hidden_layers_size,
    "activation_functions": activation_functions,
    "eta": eta,
    "eta_min": eta_min,
    "Weight_decay": Weight_decay,
    "p": p
}

# Needed Functions

In [21]:
from functions import *
from plots import *
from dataPreProcessing import *

# Plot results

In [ ]:
# Save and/or show results
show_any                = False

save_loss               = True 
show_loss               = False and show_any

save_OSP_mean_error     = True

save_histories          = True
show_histories          = False and show_any

save_histograms         = True
show_histograms         = False and show_any

save_OSP_error          = True
show_OSP_error          = False and show_any

save_times              = True

feature_names = ["sin(roll)", "cos(roll)", "sin(pitch)", "cos(pitch)", "sin(yaw)", "cos(yaw)",
                 "vx_norm", "vy_norm", "vz_norm", "w_roll_norm", "w_pitch_norm", "w_yaw_norm",
                 "thrust_norm", "torque_roll_norm", "torque_pitch_norm", "torque_yaw_norm"]

print("========== Plotting ==========")
errors_dict = {}
val_losses_dict = {}

for s, SEED in enumerate(models_dict.keys()):
    [X_train_current, X_train_next, X_train_current_norm, X_train_next_norm, 
    U_train_curr, U_train_next, U_train_curr_norm, U_train_next_norm, 
    X_val_current, X_val_next, X_val_current_norm, X_val_next_norm, 
    U_val_curr, U_val_next, U_val_curr_norm, U_val_next_norm, 
    X_test, X_test_next, U_test, U_test_next, 
    dt_mean, data_mean, data_std] = loadSeedData(SEED)
    
    print(f"\n\n==================== SEED {SEED} ====================")
    seed_dir = f"seed_{SEED}"
    save_dir = os.path.join(results_dir, file_name, activation_dir, seed_dir)
    os.makedirs(save_dir, exist_ok=True)

    print(f"Saving results to: {save_dir}")

    errors_dict[SEED] = {}
    val_losses_dict[SEED] = {}

    # Validation losses
    if save_loss or show_loss:
        print("---Calculating validation losses for all models---")
        for i, lp in enumerate(models_dict[SEED].keys()):
            val_losses_dict[SEED][lp] = {}

            clear_file = False
            if i == 0:
                clear_file = True

            val_losses = {}
            for method in models_dict[SEED][lp].keys():
                model = models_dict[SEED][lp][method]
                val_loss = validate_model(model, X_val_current_norm, U_val_curr_norm, X_val_current, X_val_next, U_val_curr, dt_mean)
                val_losses[method] = val_loss.item()

            if save_loss:
                save_val_loss(val_losses, save_dir, lp, epochs=train_epochs, NN_config=NN_config, clear_file=clear_file)

            if show_loss:
                print(f"\n---Validation Losses for L={int(lp*100)}%---")
                for method, loss in val_losses.items():
                    print(f"{method} validation loss: {loss:.10f}")

                val_loss_no_all = {method: loss for method, loss in val_losses.items() if method != "All samples"}
                print(f"\nMinimum validation loss: {min(val_loss_no_all.values()):.10f}, for model {min(val_loss_no_all, key=val_loss_no_all.get)}")
            val_losses_dict[SEED][lp] = val_losses

        print("Validation losses done.")


    # One step prediction mean error, all = True for all 12 dimensions
    if save_OSP_mean_error:
        print("---Calculating one-step prediction mean errors---")
        for i, lp in enumerate(models_dict[SEED].keys()):
            errors_dict[SEED][lp] = {}
            clear_file = False
            if i == 0:
                clear_file = True

            models = models_dict[SEED][lp]

            errors_dict[SEED][lp] = save_OSP_error_txt(models, X_test, X_test_next, U_test, dt_mean, data_mean, data_std,
                               save_dir, lp, NN_config, clear_file, train_epochs, all=False)
        print("One-step prediction mean errors done.")
    

    # Training and validation histories
    if save_histories or show_histories:
        print("---Plotting training and validation histories---")
        for i, lp in enumerate(history_dict[SEED].keys()):
            
            all_names = [method for method in history_dict[SEED][lp].keys() if history_dict[SEED][lp][method] is not None]
            all_histories = [history_dict[SEED][lp][method] for method in history_dict[SEED][lp].keys() if history_dict[SEED][lp][method] is not None]

            plot_all_histories(all_histories, all_names, save_dir, SEED, lp, save_histories, show_histories)
        print("Histories done")
    

    if save_histograms or show_histograms:
        # Histograms of the selected indices
        print("---Plotting histograms of selected indices---")
        for i, lp in enumerate(indices_dict[SEED].keys()):
            for method in indices_dict[SEED][lp].keys():
                if method == "All samples" or method == "Fisher_Random_curriculum": # Fisher random curriculum uses the same indices as random for that lp
                    continue
                plot_labeled_histograms(indices_dict[SEED][lp][method], 
                                        X_train_current_norm, U_train_curr_norm, X_train_current,
                                        method, save_dir, lp, feature_names, bins=40, 
                                        save=save_histograms, show_plot=show_histograms) 
        print("Histograms done")
    
    print("---Plotting histograms of the whole training, validation and test sets---")
    plot_set_histograms(X_val_current_norm, U_val_curr_norm, X_val_current,
                        X_train_current_norm, U_train_curr_norm, X_train_current, 
                        "Validation Set", save_dir, feature_names=None, 
                        bins=40, save=True)
    

    X_test_current_norm, U_test_current_norm = normalize_test_data(X_test, U_test, data_mean, data_std)
    X_test_current = torch.tensor(X_test, dtype=torch.float32)

    plot_set_histograms(X_test_current_norm, U_test_current_norm, X_test_current,
                            X_train_current_norm, U_train_curr_norm, X_train_current, 
                            "Test Set", save_dir, feature_names=None, 
                            bins=40, save=True)
    
    print("Histograms of whole sets done")

    # if save_OSP_error or show_OSP_error:
    #     print("---Plotting one-step predictions and errors---")
    #     for i, lp in enumerate(models_dict[SEED].keys()):
    #         models = models_dict[SEED][lp]
    #         plot_one_step(models, X_test, X_test_next, U_test, dt_mean, data_mean, data_std,
    #                       save_dir, lp,
    #                       save_OSP=False, show_plot_OSP=False, 
    #                       save_OSP_error=save_OSP_error, show_plot_OSP_error=show_OSP_error)
    #     print("One-step prediction plots and errors done.")

            
if save_loss:
    save_dir = os.path.join(results_dir, file_name, activation_dir)
    print("---Saving mean validation losses across seeds---")
    mean_val_loss(val_losses_dict, save_dir)
    print(f"\nValidation losses saved to {save_dir}")

if save_OSP_mean_error:
    save_dir = os.path.join(results_dir, file_name, activation_dir)
    print("---Saving mean one-step prediction errors across seeds---")
    mean_error(errors_dict, save_dir)
    print(f"\nOne-step prediction mean errors saved to {save_dir}")

if save_OSP_error:
    save_dir = os.path.join(results_dir, file_name, activation_dir)
    print("---Plotting one-step prediction errors across seeds---")
    plot_all_OSP(models_dict, save_dir)
    print(f"\nOne-step prediction errors saved to {save_dir}")

if save_times:
    save_dir = os.path.join(results_dir, file_name, activation_dir)
    print("---Saving sampling and training times across seeds---")
    save_timings(time_dict, save_dir)
    print(f"\nSampling and training times saved to {save_dir}")

========== Plotting ==========


==================== SEED 0 ====================
Saving results to: results\main_UAV_recreation\tanh_tanh\seed_0
---Calculating validation losses for all models---
Validation losses done.
---Calculating one-step prediction mean errors---
One-step prediction mean errors done.
---Plotting training and validation histories---
Histories done
---Plotting histograms of selected indices---


c:\Users\emils\Documents\Bachelor\BachelorProject\FinalFolder UAV\plots.py:125: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


Histograms done


C:\Users\emils\AppData\Local\Temp\ipykernel_1704\3644706523.py:60: UserWarning: The figure layout has changed to tight
  plt.tight_layout()




==================== SEED 1 ====================
Saving results to: results\main_UAV_recreation\tanh_tanh\seed_1
---Calculating validation losses for all models---
Validation losses done.
---Calculating one-step prediction mean errors---
One-step prediction mean errors done.
---Plotting training and validation histories---
Histories done
---Plotting histograms of selected indices---
Histograms done

Validation losses saved to results\main_UAV_recreation\tanh_tanh

One-step prediction mean errors saved to results\main_UAV_recreation\tanh_tanh

One-step prediction errors saved to results\main_UAV_recreation\tanh_tanh
